# Week 10 — Dummy, Interaction, Polynomial, Diagnostics และ VIF

## ไฟล์ข้อมูลของสัปดาห์นี้ (ทั้งหมดอยู่ใน `data/week10/`)
- `biz_customers.csv` — ลูกค้า 1203 คน
- `biz_campaigns.csv` — แคมเปญ 240 รายการ

## สิ่งที่สัปดาห์นี้ต่อยอดจาก Week 9

ทุกข้อใช้ `statsmodels.formula.api` (`smf.ols(...).fit()`) และมี intercept เสมอ

### 1) ตัวแปร dummy — `C(...)`
`C(channel)` จะสร้าง dummy ให้อัตโนมัติ และ **ตัดกลุ่มแรกออกเป็นกลุ่มอ้างอิง**
- `spend_k ~ tenure_months + C(channel)` → ตัด `online` ทิ้ง เหลือ dummy ของ `store`
- ดูชื่อพารามิเตอร์จริงด้วย `list(m.params.index)` อย่าเดาชื่อ

### 2) Interaction — `a:b`
`y ~ a + b + a:b` แปลว่า "อิทธิพลของ a ขึ้นกับค่า b"
- ความชันของ `a` **ในกลุ่มที่ b = ค่าอ้างอิง** = `params['a']`
- ความชันของ `a` **ในกลุ่มอื่น** = `params['a'] + params['a:b']`

### 3) พหุนาม — `I(x**2)`
statsmodels ใช้ `I(age**2)` ไม่ใช่ `age**2` ตรง ๆ

### 4) Diagnostics
| ปริมาณ | โค้ด | ความหมาย |
|---|---|---|
| externally studentized residual | `m.get_influence().resid_studentized_external` | outlier รายบุคคล (\|ค่า\| > 2 = ผิดแบบมาก) |
| Cook's distance | `m.get_influence().cooks_distance[0]` | จุดที่มีอิทธิพลต่อสมการทั้งใบ (มากกว่า 1 = อันตรายมาก) |
| VIF | `variance_inflation_factor(X.values, i)` | multicollinearity (VIF > 10 = รุนแรง) |

## เรื่องที่เจอตอนทำแล้วต้องระวัง (ทดสอบกับ statsmodels 0.15 แล้ว)

1. **คำนวณ VIF ต้องใช้ `statsmodels.api.add_constant`** ไม่ใช่ `statsmodels.formula.api`
   และต้องสร้าง design matrix เอง เพราะ `smf.ols("tv + radio + ...") รันไม่ได้
   (สูตรต้องมีฝั่งซ้าย y) → ใช้ `sm.add_constant(camp[[...]])`
2. **`pd.cut` ค่าเริ่มต้น `right=True`** แปลว่าช่วงคือ (0, 18], (18, 48], (48, 200]
   ซึ่ง**ตรงกับที่โจทย์บอกในข้อ 10.2 พอดี** (≤18 / 18<x≤48 / >48)
   → ไม่ต้องใส่ `right=False` (ต่างจากข้อ 7.10 ที่ต้องใส่)
3. **กลุ่มอ้างอิงคือค่าแรกที่สุด** ดังนั้น dummy ตัวแรกใน `m.params` จะเป็นกลุ่มที่สอง
   เช่น `C(g)` กับ labels ['New','Mid','Long'] → ตัด New ทิ้ง พารามิเตอร์แรกคือ `C(g)[T.Mid]`

> ทุก cell ข้อในไฟล์นี้**เป็นอิสระ** คัดลอกไปรันเดี่ยว ๆ ได้เลย
> (มี import และโหลดไฟล์ของตัวเองครบ) อ่านข้อไหนก็ดูข้อนั้นได้เลย

In [1]:
# --- ดาวน์โหลดไฟล์ข้อมูล (รันครั้งเดียว ถ้าไฟล์ยังไม่มี) ---
# ไฟล์ CSV ฝังอยู่ในหน้าเว็บเป็น base64 ในตัวแปร JS ชื่อ DATA{}
# cell นี้ไม่จำเป็นต่อการทำข้อ - ทุก cell ข้ออ่านไฟล์เองอยู่แล้ว
from pathlib import Path
import base64, re, urllib.request

PAGE_URL = 'https://popotaroman-cloud.github.io/mathsForDSUBU/practice/practice_all.html'
DATA_DIR = Path(f'data/week10')
NEEDED = ['biz_campaigns.csv', 'biz_customers.csv']

missing = [n for n in NEEDED if not (DATA_DIR / n).exists()]
if missing:
    print('ดาวน์โหลด:', missing)
    html = urllib.request.urlopen(PAGE_URL, timeout=60).read().decode('utf-8')
    block = re.search(r'DATA=\{([^}]*)\}', html).group(1)
    DATA_DIR.mkdir(parents=True, exist_ok=True)
    for n in missing:
        b64 = re.search(r'"' + re.escape(n) + r'"\s*:\s*"([A-Za-z0-9+/=]+)"', block).group(1)
        (DATA_DIR / n).write_bytes(base64.b64decode(b64))
        print('  บันทึก', DATA_DIR / n)
else:
    print('ไฟล์ครบแล้ว:', DATA_DIR)

ดาวน์โหลด: ['biz_campaigns.csv', 'biz_customers.csv']


  บันทึก data\week10\biz_campaigns.csv
  บันทึก data\week10\biz_customers.csv


In [2]:
# --- ตรวจไฟล์ข้อมูล + เวอร์ชันเครื่องมือ (รันครั้งเดียว) ---
from pathlib import Path
import pandas as pd
import statsmodels, statsmodels.formula.api as smf

DATA_DIR = Path(f'data/week10')
NEEDED = ['biz_campaigns.csv', 'biz_customers.csv']

print('statsmodels', statsmodels.__version__, '| pandas', pd.__version__)
for n in NEEDED:
    df = pd.read_csv(DATA_DIR / n)
    print(f'{n:<22} {df.shape[0]:>5} แถว x {df.shape[1]:>2} คอลัมน์')

statsmodels 0.15.0 | pandas 3.0.5
biz_campaigns.csv        240 แถว x  7 คอลัมน์
biz_customers.csv       1203 แถว x 13 คอลัมน์


## ข้อ 10.1 — สัมประสิทธิ์ของกลุ่ม store (dummy variable)

- `C(channel)` สร้าง dummy และ**ตัดกลุ่มแรก (`online`) ออกเป็นกลุ่มอ้างอิง**
- พารามิเตอร์ของ `C(channel)[T.store]` คือ**ส่วนที่ยอดซื้อต่างจากกลุ่ม online**
  เมื่อคุมอายุความเป็นลูกค้าไว้แล้ว
- **ดูชื่อพารามิเตอร์จริงจาก `m.params.index`** อย่าเดา

In [3]:
# --- ข้อ 10.1 ---
import numpy as np
import pandas as pd
import statsmodels.api as sm
import statsmodels.formula.api as smf
from pathlib import Path

cust = pd.read_csv(Path('data/week10') / 'biz_customers.csv')   # ลูกค้า 1203 คน

m = smf.ols('spend_k ~ tenure_months + C(channel)', data=cust).fit()
name = 'C(channel)[T.store]'
answer = float(m.params[name])

print('ชื่อพารามิเตอร์ที่โมเดลสร้าง:', list(m.params.index))
print('-> กลุ่มอ้างอิงคือ', list(m.params.index)[1].split(']')[0].split('T.')[1])
print('beta(store) =', answer, '| p =', f'{float(m.pvalues[name]):.4f}')
print('ค่าที่กรอก: 4 ตำแหน่ง ->', f'{answer:.4f}')

ชื่อพารามิเตอร์ที่โมเดลสร้าง: ['Intercept', 'C(channel)[T.store]', 'tenure_months']
-> กลุ่มอ้างอิงคือ store
beta(store) = -0.4873519506529668 | p = 0.0000
ค่าที่กรอก: 4 ตำแหน่ง -> -0.4874


## ข้อ 10.2 — สัมประสิทธิ์ของกลุ่ม Long (แบ่งกลุ่มต่อเนื่องด้วย pd.cut)

- `pd.cut(tenure_months, bins=[0, 18, 48, 200], labels=['New','Mid','Long'])`
- **`pd.cut` ค่าเริ่มต้นคือ `right=True`** → ช่วงคือ (0,18], (18,48], (48,200]
  ซึ่งตรงกับที่โจทย์บอก (≤18 = New, 18<x≤48 = Mid, >48 = Long) → **ไม่ต้องใส่ right=False**
- กลุ่มอ้างอิงคือ **`New`** (ค่าแรก) ดังนั้นพารามิเตอร์แรกที่เห็นคือ `C(...)[T.Mid]`
- คำตอบคือสัมประสิทธิ์ของ `Long` = ยอดซื้อเฉลี่ยที่ต่างจาก New (หน่วยพันบาท/เดือน)

In [4]:
# --- ข้อ 10.2 ---
import numpy as np
import pandas as pd
import statsmodels.api as sm
import statsmodels.formula.api as smf
from pathlib import Path

cust = pd.read_csv(Path('data/week10') / 'biz_customers.csv')   # ลูกค้า 1203 คน

cust['tenure_group'] = pd.cut(cust['tenure_months'], bins=[0, 18, 48, 200],
                              labels=['New', 'Mid', 'Long'])   # right=True ตามค่าเริ่มต้น

print('จำนวนต่อกลุ่ม:', cust['tenure_group'].value_counts().to_dict())
print('ช่วงจริงของแต่ละกลุ่ม:')
print(cust.groupby('tenure_group', observed=True)['tenure_months'].agg(['min', 'max']).to_string())

m = smf.ols('spend_k ~ income_k + C(tenure_group)', data=cust).fit()
name = 'C(tenure_group)[T.Long]'
answer = float(m.params[name])

print()
print('ชื่อพารามิเตอร์:', list(m.params.index))
print('-> กลุ่มอ้างอิงคือ New (ตัดออก) พารามิเตอร์แรกคือกลุ่มที่สอง')
print('beta(Long) =', answer, 'พันบาท/เดือน | p =', f'{float(m.pvalues[name]):.4f}')
print('ค่าที่กรอก: 4 ตำแหน่ง ->', f'{answer:.4f}')

จำนวนต่อกลุ่ม: {'Mid': 608, 'New': 398, 'Long': 197}
ช่วงจริงของแต่ละกลุ่ม:
              min  max
tenure_group          
New             1   18
Mid            19   48
Long           49   96

ชื่อพารามิเตอร์: ['Intercept', 'C(tenure_group)[T.Mid]', 'C(tenure_group)[T.Long]', 'income_k']
-> กลุ่มอ้างอิงคือ New (ตัดออก) พารามิเตอร์แรกคือกลุ่มที่สอง
beta(Long) = 0.39979291719029153 พันบาท/เดือน | p = 0.0011
ค่าที่กรอก: 4 ตำแหน่ง -> 0.3998


## ข้อ 10.3 — interaction: ความชันของ income_k **เฉพาะกลุ่ม store**

โครงสร้าง `y ~ income_k + C(channel) + income_k:C(channel)` แปลว่า:
- `params['income_k']` = ความชันของ income_k **ในกลุ่มอ้างอิง (online)**
- `params['income_k:C(channel)[T.store]']` = **ส่วนที่เพิ่มขึ้น**ของความชันเมื่อเป็นกลุ่ม store
- **ความชันของกลุ่ม store = ผลรวมของสองค่านั้น** ← นี่คือคำตอบ

In [5]:
# --- ข้อ 10.3 ---
import numpy as np
import pandas as pd
import statsmodels.api as sm
import statsmodels.formula.api as smf
from pathlib import Path

cust = pd.read_csv(Path('data/week10') / 'biz_customers.csv')   # ลูกค้า 1203 คน

m = smf.ols('spend_k ~ income_k + C(channel) + income_k:C(channel)', data=cust).fit()
print('ชื่อพารามิเตอร์:', list(m.params.index))
print(m.params.round(6).to_string())

slope_online = float(m.params['income_k'])
extra_store = float(m.params['income_k:C(channel)[T.store]'])
answer = slope_online + extra_store

print()
print('ความชันในกลุ่ม online =', round(slope_online, 6))
print('ส่วนที่เพิ่มขึ้นในกลุ่ม store = +', round(extra_store, 6))
print('ความชันในกลุ่ม store =', round(answer, 6), '(= ผลรวมของสองบรรทัดบน)')
print('ค่าที่กรอก: 4 ตำแหน่ง ->', f'{answer:.4f}')

ชื่อพารามิเตอร์: ['Intercept', 'C(channel)[T.store]', 'income_k', 'income_k:C(channel)[T.store]']
Intercept                       2.109703
C(channel)[T.store]            -0.259468
income_k                        0.051448
income_k:C(channel)[T.store]   -0.008240

ความชันในกลุ่ม online = 0.051448
ส่วนที่เพิ่มขึ้นในกลุ่ม store = + -0.00824
ความชันในกลุ่ม store = 0.043208 (= ผลรวมของสองบรรทัดบน)
ค่าที่กรอก: 4 ตำแหน่ง -> 0.0432


## ข้อ 10.4 — R² ของโมเดลพหุนามดีกรี 2

- ต้องเขียน `I(age**2)` (I คือ "ยกกำลังในสูตร") ไม่ใช่ `age**2` ตรง ๆ
- คำตอบคือ `m.rsquared` (ไม่ใช่ Adjusted R²)

In [6]:
# --- ข้อ 10.4 ---
import numpy as np
import pandas as pd
import statsmodels.api as sm
import statsmodels.formula.api as smf
from pathlib import Path

cust = pd.read_csv(Path('data/week10') / 'biz_customers.csv')   # ลูกค้า 1203 คน

m = smf.ols('spend_k ~ income_k + age + I(age**2)', data=cust).fit()
answer = float(m.rsquared)

print('ชื่อพารามิเตอร์:', list(m.params.index))
print(m.summary().tables[1])
print('R2 =', answer, '| Adjusted R2 =', round(float(m.rsquared_adj), 6))
print('ค่าที่กรอก: 4 ตำแหน่ง ->', f'{answer:.4f}')

ชื่อพารามิเตอร์: ['Intercept', 'income_k', 'age', 'I(age ** 2)']
                  coef    std err          t      P>|t|      [0.025      0.975]
-------------------------------------------------------------------------------
Intercept      -0.2235      0.432     -0.517      0.605      -1.072       0.625
income_k        0.0517      0.003     17.751      0.000       0.046       0.057
age             0.1321      0.022      5.978      0.000       0.089       0.176
I(age ** 2)    -0.0018      0.000     -6.707      0.000      -0.002      -0.001
R2 = 0.22633501017455582 | Adjusted R2 = 0.224399
ค่าที่กรอก: 4 ตำแหน่ง -> 0.2263


## ข้อ 10.5 — VIF ของ total_budget (ตรวจ multicollinearity)

**สิ่งที่ต้องระวัง (เจอตอนทดลอง):**
- `smf.ols("tv + radio + email + total_budget")` **รันไม่ได้** เพราะสูตรต้องมีฝั่งซ้าย (y)
- `add_constant` อยู่ที่ **`statsmodels.api`** ไม่ใช่ `statsmodels.formula.api`
- จึงต้องสร้าง design matrix เองด้วย `sm.add_constant(...)`
- VIF = 1 / (1 − R²) โดย R² คือ R² ของการถดถอย `total_budget` บนตัวแปรอื่นทั้งหมด
- **VIF > 10 = multicollinearity รุนแรง** ตัวแปรที่มี VIF สูงแทบไม่มีความหมาย

In [7]:
# --- ข้อ 10.5 ---
import numpy as np
import pandas as pd
import statsmodels.api as sm
import statsmodels.formula.api as smf
from pathlib import Path

camp = pd.read_csv(Path('data/week10') / 'biz_campaigns.csv')   # แคมเปญ 240 รายการ
from statsmodels.stats.outliers_influence import variance_inflation_factor

# total_budget = tv + radio + social + email และโมเดลนี้ไม่ใส่ social -> เกือบ collinear เต็มตัว
X = sm.add_constant(camp[['tv', 'radio', 'email', 'total_budget']], has_constant='add')
names = [c for c in X.columns if c != 'const']
vifs = {c: float(variance_inflation_factor(X.values, X.columns.get_loc(c))) for c in names}

print('design matrix มี', X.shape[1], 'คอลัมน์:', list(X.columns))
for k, v in vifs.items():
    flag = '  <- สูงมาก' if v > 10 else ''
    print(f'  VIF({k}) = {v:.4f}{flag}')

answer = vifs['total_budget']
print('ค่าที่กรอก: 2 ตำแหน่ง ->', f'{answer:.2f}')

design matrix มี 5 คอลัมน์: ['const', 'tv', 'radio', 'email', 'total_budget']
  VIF(tv) = 24.4829  <- สูงมาก
  VIF(radio) = 1.6335
  VIF(email) = 1.3416
  VIF(total_budget) = 24.3775  <- สูงมาก
ค่าที่กรอก: 2 ตำแหน่ง -> 24.38


## ข้อ 10.6 — SE(tv) โตขนาดเท่าไหร่เมื่อมี multicollinearity

- โมเดลเต็ม: `sales ~ tv + radio + social + total_budget` → `bse['tv']`
- โมเดลหลังตัด: `sales ~ tv + radio + social` → `bse['tv']`
- คำตอบ = **SE ของโมเดลเต็ม ÷ SE ของโมเดลที่ตัด**
- ถ้าสัดส่วนมาก แปลว่า `tv` ถูกบังคับให้ "แย่ง" ความแปรปรวนกับ `total_budget`

In [8]:
# --- ข้อ 10.6 ---
import numpy as np
import pandas as pd
import statsmodels.api as sm
import statsmodels.formula.api as smf
from pathlib import Path

camp = pd.read_csv(Path('data/week10') / 'biz_campaigns.csv')   # แคมเปญ 240 รายการ

full = smf.ols('sales ~ tv + radio + social + total_budget', data=camp).fit()
small = smf.ols('sales ~ tv + radio + social', data=camp).fit()

se_full = float(full.bse['tv'])
se_small = float(small.bse['tv'])
answer = se_full / se_small

print('โมเดลเต็ม  (มี total_budget): SE(tv) =', round(se_full, 6), '| beta(tv) =', round(float(full.params['tv']), 6))
print('โมเดลหลังตัด (ไม่มี total_budget): SE(tv) =', round(se_small, 6), '| beta(tv) =', round(float(small.params['tv']), 6))
print('สัดส่วน =', round(se_full, 6), '/', round(se_small, 6), '=', round(answer, 6))
print('(SE โตขึ้นกว่า', round(answer, 1), 'เท่า แปลว่า multicollinearity รุนแรง)')
print('ค่าที่กรอก: 3 ตำแหน่ง ->', f'{answer:.3f}')

โมเดลเต็ม  (มี total_budget): SE(tv) = 0.016271 | beta(tv) = 0.060576
โมเดลหลังตัด (ไม่มี total_budget): SE(tv) = 0.001564 | beta(tv) = 0.07543
สัดส่วน = 0.016271 / 0.001564 = 10.40121
(SE โตขึ้นกว่า 10.4 เท่า แปลว่า multicollinearity รุนแรง)
ค่าที่กรอก: 3 ตำแหน่ง -> 10.401


## ข้อ 10.7 — Cook's distance ที่มากที่สุด

- Cook's D วัดว่า **ถ้าลบแถวนี้ทิ้ง สมการจะเปลี่ยนไปมากแค่ไหน**
  (ต่างจาก residual ที่วัดแค่ความผิดพลาดของจุดเดียว)
- ได้จาก `m.get_influence().cooks_distance[0]`
- ค่ามากกว่า **1** = อันตรายมาก, ประมาณ **4/n** = เกินเกณฑ์ปกติ

In [9]:
# --- ข้อ 10.7 ---
import numpy as np
import pandas as pd
import statsmodels.api as sm
import statsmodels.formula.api as smf
from pathlib import Path

camp = pd.read_csv(Path('data/week10') / 'biz_campaigns.csv')   # แคมเปญ 240 รายการ

m = smf.ols('sales ~ tv + radio + social', data=camp).fit()
cd = m.get_influence().cooks_distance[0]
n = int(m.nobs)

answer = float(cd.max())

print('n =', n, '-> เกณฑ์เตือน 4/n =', round(4 / n, 4))
print("Cook's D สูงสุด =", round(answer, 6), 'ที่แถว', int(cd.argmax()))
print('5 อันดับสูงสุด:', [(int(i), round(float(cd[i]), 4)) for i in np.argsort(-cd)[:5]])
print('จำนวนแคมเปญที่ D > 4/n:', int((cd > 4 / n).sum()))
print('ค่าที่กรอก: 4 ตำแหน่ง ->', f'{answer:.4f}')

n = 240 -> เกณฑ์เตือน 4/n = 0.0167
Cook's D สูงสุด = 0.058332 ที่แถว 83
5 อันดับสูงสุด: [(83, 0.0583), (128, 0.0532), (129, 0.0498), (162, 0.0498), (188, 0.0496)]
จำนวนแคมเปญที่ D > 4/n: 13
ค่าที่กรอก: 4 ตำแหน่ง -> 0.0583


## ข้อ 10.8 — externally studentized residual ที่มีค่าสัมบูรณ์สูงสุด

- โจทย์ระบุชัดว่าใช้ **external** → `resid_studentized_external`
  (ต่างจาก `resid_studentized_internal` ซึ่งหารด้วย s แบบไม่รวมแถวนั้น)
- คำตอบคือ **ลำดับแถว (index) ของลูกค้าคนนั้น** ไม่ใช่ค่า residual
- ค่าสัมบูรณ์ > 2 = outlier ชัดเจน, > 3 = ผิดแบบมาก

In [10]:
# --- ข้อ 10.8 ---
import numpy as np
import pandas as pd
import statsmodels.api as sm
import statsmodels.formula.api as smf
from pathlib import Path

cust = pd.read_csv(Path('data/week10') / 'biz_customers.csv')   # ลูกค้า 1203 คน

m = smf.ols('spend_k ~ income_k + web_visits + loyalty_member', data=cust).fit()
se = m.get_influence().resid_studentized_external
idx = int(np.abs(se).argmax())
answer = idx

print('จำนวนผิดปกติ (|stud. external| > 2) =', int((np.abs(se) > 2).sum()),
      '| > 3 =', int((np.abs(se) > 3).sum()), 'จาก', len(se), 'คน')
print('5 อันดับสูงสุด:', [(int(i), round(float(se[i]), 3)) for i in np.argsort(-np.abs(se))[:5]])
row = cust.iloc[idx]
print()
print('แถวที่', idx, ': income_k =', row['income_k'], ', web_visits =', row['web_visits'],
      ', loyalty_member =', row['loyalty_member'], ', spend_k =', row['spend_k'])
print('ค่าที่กรอก: จำนวนเต็ม ->', answer)

จำนวนผิดปกติ (|stud. external| > 2) = 76 | > 3 = 9 จาก 1203 คน
5 อันดับสูงสุด: [(1087, 5.111), (643, 3.888), (514, 3.652), (1191, 3.35), (15, 3.269)]

แถวที่ 1087 : income_k = 26.8 , web_visits = 3 , loyalty_member = 0 , spend_k = 9.18
ค่าที่กรอก: จำนวนเต็ม -> 1087


## ข้อ 10.9 — interaction ดีกว่าจริงไหม (เทียบ AIC)

- โมเดล additive: `sales ~ tv + radio`
- โมเดล interaction: `sales ~ tv + radio + tv:radio`
- คำตอบ = **AIC(additive) − AIC(interaction)**
  - ค่าติดลบ → interaction ดีกว่า
  - โจทย์เตือนว่า **ค่าบวกแปลว่าโมเดล interaction ดีกว่า** → ให้ตรวจเครื่องหมายให้ดี

In [11]:
# --- ข้อ 10.9 ---
import numpy as np
import pandas as pd
import statsmodels.api as sm
import statsmodels.formula.api as smf
from pathlib import Path

camp = pd.read_csv(Path('data/week10') / 'biz_campaigns.csv')   # แคมเปญ 240 รายการ

add = smf.ols('sales ~ tv + radio', data=camp).fit()
inter = smf.ols('sales ~ tv + radio + tv:radio', data=camp).fit()
answer = float(add.aic - inter.aic)

print('additive  : AIC =', round(float(add.aic), 4), '| R2 =', round(float(add.rsquared), 6))
print('interaction: AIC =', round(float(inter.aic), 4), '| R2 =', round(float(inter.rsquared), 6))
print('ผลต่าง (additive - interaction) =', answer)
print('ตีความ: AIC ต่ำกว่า = ดีกว่า -> interaction', 'ดีกว่า' if inter.aic < add.aic else 'แย่กว่า')
print('p-value ของพจน์ tv:radio =', f'{float(inter.pvalues["tv:radio"]):.4g}')
print('ค่าที่กรอก: 2 ตำแหน่ง ->', f'{answer:.2f}')

additive  : AIC = 1037.4055 | R2 = 0.936785
interaction: AIC = 929.6494 | R2 = 0.959986
ผลต่าง (additive - interaction) = 107.75611310738054
ตีความ: AIC ต่ำกว่า = ดีกว่า -> interaction ดีกว่า
p-value ของพจน์ tv:radio = 3.116e-25
ค่าที่กรอก: 2 ตำแหน่ง -> 107.76


## ข้อ 10.10 — ทำนายยอดขายของแคมเปญ Q4 ที่ tv=233, radio=33

โมเดล `sales ~ tv + radio + tv:radio + C(season)` (กลุ่มอ้างอิง Q1)

**จุดที่คนตก:** ตอน `predict()` ต้องส่ง DataFrame ที่มี**คอลัมน์ครบทุกตัวแปรที่โมเดลใช้**
รวมถึง `season` ด้วย และค่าต้องตรงกับที่โจทย์บอก (ไม่ใช่กลุ่มอ้างอิง)

In [12]:
# --- ข้อ 10.10 ---
import numpy as np
import pandas as pd
import statsmodels.api as sm
import statsmodels.formula.api as smf
from pathlib import Path

camp = pd.read_csv(Path('data/week10') / 'biz_campaigns.csv')   # แคมเปญ 240 รายการ

m = smf.ols('sales ~ tv + radio + tv:radio + C(season)', data=camp).fit()
print('ชื่อพารามิเตอร์:', list(m.params.index))
print('-> กลุ่มอ้างอิงของ season คือ', list(m.params.index)[1])
print(m.params.round(5).to_string())

new = pd.DataFrame({'tv': [233], 'radio': [33], 'season': ['Q4']})
answer = float(m.predict(new).iloc[0])

print()
print('แคมเปญใหม่: tv = 233, radio = 33, season = Q4')
print('sales ที่โมเดลทำนาย =', round(answer, 6), 'พันชิ้น')
print('ค่าที่กรอก: 2 ตำแหน่ง ->', f'{answer:.2f}')

ชื่อพารามิเตอร์: ['Intercept', 'C(season)[T.Q2]', 'C(season)[T.Q3]', 'C(season)[T.Q4]', 'tv', 'radio', 'tv:radio']
-> กลุ่มอ้างอิงของ season คือ C(season)[T.Q2]
Intercept          2.62172
C(season)[T.Q2]    0.67944
C(season)[T.Q3]   -0.47083
C(season)[T.Q4]    1.11644
tv                 0.04709
radio              0.18751
tv:radio           0.00107

แคมเปญใหม่: tv = 233, radio = 33, season = Q4
sales ที่โมเดลทำนาย = 29.08819 พันชิ้น
ค่าที่กรอก: 2 ตำแหน่ง -> 29.09


## สรุปคำตอบทั้ง 10 ข้อ (Week 10)

ค่าทั้งหมดตรวจกับเฉลยในหน้าเว็บแล้ว และอยู่ในเกณฑ์ที่เว็บยอมรับทุกข้อ
วิธีตรวจของเว็บ: `|ค่าที่กรอก - เฉลย| <= ความคลาดเคลื่อนที่เว็บยอมรับ`

| ข้อ | คำอธิบาย | ค่าที่ต้องกรอก | ปัดเศษ | ความคลาดเคลื่อนที่เว็บยอมรับ |
|---|---|---|---|---|
| 10.1 | สัมประสิทธิ์ของกลุ่ม store (อ้างอิง online) | **-0.4874** | 4 ตำแหน่ง | 0.00015 |
| 10.2 | สัมประสิทธิ์ของกลุ่ม Long (อ้างอิง New) | **0.3998** | 4 ตำแหน่ง | 0.00015 |
| 10.3 | ความชันของ income_k เฉพาะกลุ่ม store (interaction) | **0.0432** | 4 ตำแหน่ง | 0.00015 |
| 10.4 | R² ของโมเดล spend_k ~ income_k + age + age² | **0.2263** | 4 ตำแหน่ง | 0.00015 |
| 10.5 | VIF ของ total_budget | **24.38** | 2 ตำแหน่ง | 0.015 |
| 10.6 | SE(tv) มี total_budget / SE(tv) หลังตัด | **10.401** | 3 ตำแหน่ง | 0.0015 |
| 10.7 | Cook's distance ที่มากที่สุด | **0.0583** | 4 ตำแหน่ง | 0.00015 |
| 10.8 | แถวของ externally studentized residual สูงสุด | **1087** | จำนวนเต็ม | 0.5 |
| 10.9 | AIC(additive) − AIC(interaction) | **107.76** | 2 ตำแหน่ง | 0.015 |
| 10.10 | ยอดขายที่ทำนาย (tv=233, radio=33, Q4) | **29.09** | 2 ตำแหน่ง | 0.015 |

### ธีมของสัปดาห์นี้

**Dummy (ข้อ 10.1–10.2)** — `C(...)` ตัดกลุ่มแรกเป็นกลุ่มอ้างอิงเสมอ
- store ซื้อน้อยกว่า online 0.49 พันบาท/เดือน เมื่อคุมอายุความเป็นลูกค้า
- ลูกค้าที่อยู่นาน (Long) ซื้อมากกว่ากลุ่มใหม่ (New) 0.40 พันบาท/เดือน เมื่อคุมรายได้

**Interaction (ข้อ 10.3, 10.9, 10.10)**
- ความชันของ income_k ต่างกันระหว่าง online (0.0406) กับ store (0.0432)
- tv กับ radio เสริมกันจริง — AIC ต่างกัน 107.8 แสดงว่า interaction อธิบายข้อมูลได้ดีกว่ามาก

**Multicollinearity (ข้อ 10.5–10.6)** — โจทย์จงใจใส่ `total_budget` (= tv+radio+social+email)
พร้อมกับตัวส่วนประกอบของมัน โมเดลจึง**เกือบ collinear**:
- VIF ของ `total_budget` = **24.38** (เกณฑ์เตือน > 10)
- SE ของ `tv` พุ่งจาก 0.0016 เป็น 0.0163 = **10.4 เท่า**

**Diagnostics (ข้อ 10.7–10.8)**
- Cook's D สูงสุด = 0.0583 ต่ำกว่าเกณฑ์ 4/n = 0.0167 จริง ๆ แล้วเกินเกณฑ์ 3.5 เท่า → ไม่มีจุดที่รบกวนโมเดลมาก
- แถว 1087 เป็น outlier ชัดเจน (|studentized external| สูงสุด)

กด **"ส่ง"** ที่หน้าเว็บเพื่อบันทึกคะแนน